# Price-Trend CNN — extra experiments

Everything lives in **`price_trend_cnn.py`**; this notebook just imports it and flips knobs.
Every run saves, into a `results_<tag>/` folder next to the data:

- `predictions.csv` — per (stock, date) OOS score + realised return  **and the trained model** under `model/`
- `portfolio_period_returns.csv` — long-short (H–L) & long-only period returns **with the exact `rebalance_date`**
- `beta_vs_time.csv/.png` — half-year regression of realised return on the (standardised) score
- `decile_profile.csv/.png` — average realised return of each score-decile over the whole OOS
- `portfolio_cumulative.png`, `summary.json`

**No look-ahead:** test formation is strictly after the train label window (+embargo); the pixel/y/VIX
scalers and any QC threshold are fit on **train only**; labels are strictly-forward returns.


In [ ]:
%load_ext autoreload
%autoreload 2
import price_trend_cnn as pt
import pandas as pd
print('module :', pt.__file__)
print('presets:', list(pt.EXPERIMENTS))


## Config

Point `DATA_DIR` at the folder holding your chart pickles:
**`SP500_pic_Map5MA_price_chart_image_df.pickle`**, **`SP600_pic_Map5MA_...`**, **`SP500_pic_Map20MA_...`**.
The presets refer to them by short tags (`SP500_5MA`, `SP600_5MA`, `SP500_20MA`) that resolve
**case-insensitively** (a `5MA` tag never grabs a `20MA` file). Here `5MA`/`20MA` = the image
**look-back** length (5-day / 20-day); `horizon` is the separate forward holding period.
The VIX test additionally needs a `vix.csv` (auto-downloaded via yfinance if absent).


In [ ]:
DATA_DIR = '.'   # folder with SP500_pic_Map5MA / SP600_pic_Map5MA / SP500_pic_Map20MA (+ vix.csv for the VIX test)

# --- OFFLINE dry-run? synthesize schema-faithful fixtures with the SAME filenames: ---
# pt.make_synth_suite(data_dir=DATA_DIR, years=22)   # writes SP500/SP600 5MA, SP500 20MA + vix.csv

pt.EXPERIMENTS   # the 6 presets (baseline + the 5 distinct test configs)


## Run one experiment
`run_experiment(name)` = the preset; append any `run()` kwarg to override.


In [ ]:
res = pt.run_experiment('baseline_sp500_5MA', data_dir=DATA_DIR)
print(res['summary'])


In [ ]:
res['portfolio'].head()        # long_short / long_only per period WITH rebalance_date


In [ ]:
res['beta'].head()             # half-year beta of realised return on the score


In [ ]:
res['decile_profile']          # avg realised return per score decile (whole OOS)


In [ ]:
print('all files saved in:', res['out_dir'])


## The eight tests

| # | what | how |
|---|------|-----|
| 1 | train **SP500** → test **SP600** (time-aligned, strictly OOS) | `cross_sp500_to_sp600` |
| 2 | save predictions **and** the model | on for every run (`save_csv`, `save_model`) |
| 3 | half-year score→return beta vs time | `res['beta']` + `beta_vs_time.png` |
| 4 | portfolio period returns w/ rebalance dates + decile-return graph | `portfolio_period_returns.csv`, `decile_profile.png` |
| 5 | append a VIX strip under the volume block (own scaler) | `vix_sp500_5MA` (`add_vix=True`) |
| 6 | per-image normalisation (vs global) | `perimg_sp500_5MA` (`pixel_norm='per_image'`) |
| 7 | 15 years of training | `train15y_sp500_5MA` (`train_years=15`) |
| 8 | 20-day look-back image, 5-day horizon (weekly) | `sp500_20MA_h5` |


In [ ]:
# (1) cross-universe: train on SP500, test on SP600 (test forced strictly after the train cut)
res_cross = pt.run_experiment('cross_sp500_to_sp600', data_dir=DATA_DIR)
res_cross['summary']


In [ ]:
# (5) VIX channel: +vix_rows rows of a column-aligned, separately-scaled VIX strip (scaler fit on train)
res_vix = pt.run_experiment('vix_sp500_5MA', data_dir=DATA_DIR)   # vix_rows=3, vix_scale='minmax'
res_vix['summary']


In [ ]:
# (6) per-image pixel normalisation instead of the global train mean/std
res_pi = pt.run_experiment('perimg_sp500_5MA', data_dir=DATA_DIR)
res_pi['summary']


In [ ]:
# (7) 15-year training window
res_15 = pt.run_experiment('train15y_sp500_5MA', data_dir=DATA_DIR)
res_15['summary']


In [ ]:
# (8) 20-day look-back image with a 5-day horizon -> weekly rebalance
res_205 = pt.run_experiment('sp500_20MA_h5', data_dir=DATA_DIR)
res_205['summary']


## Override any knob
Presets are just `dict`s of `run()` kwargs; override inline.
Useful knobs: `task='clf'|'reg'`, `horizon=5|20|60`, `pixel_norm='global'|'per_image'`,
`add_vix`, `vix_rows`, `vix_scale='minmax'|'zscore'`, `train_years`, `seeds`, `epochs`,
`test_start`, `test_end`, `beta_freq_months`, `n_decile_bins`.


In [ ]:
# e.g. VIX with 5 rows + z-score scaler, regression head, faster ensemble
# r = pt.run_experiment('vix_sp500_5MA', data_dir=DATA_DIR, vix_rows=5, vix_scale='zscore', task='reg', seeds=3)


## Reload a saved model
Every run persists the ensemble (state-dicts) + scalers; reload and score any image cube.


In [ ]:
pred = pt.load_predictor(res['out_dir'] + '/model', 'model')
# scores = pred(X)   # X: (N, H, W) float32 cube, same geometry the model was trained on


## Run everything (heavy)


In [ ]:
# summary = pt.run_all(data_dir=DATA_DIR)
# pd.DataFrame(summary).T
